# 01 - Querying Hetionet with Cypher over the Neo4j HTTP API

Hetionet is a biomedical knowledge graph on a public, read-only Neo4j 3.5.12 server. This notebook shows how to send Cypher queries to it with `httpx`: first as a raw HTTP call, then through the project wrapper `src/db.py`, and finally how to check a query with `EXPLAIN` without running it.

The notebook sends 10 small read-only requests. No login or API key is needed.

In [1]:
import os
import sys
from pathlib import Path

# Find the repository root (the folder with pyproject.toml), wherever the notebook is started from.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
os.chdir(ROOT)                              # project code uses paths relative to the repo root
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))   # modules in src/ import each other flat: `from db import run_cypher`
print("repo root:", ROOT.name)

repo root: reproducible-agents-biomedical-kg


## 1. One raw HTTP call

The transactional endpoint takes a POST with the body `{"statements": [{"statement": "<cypher>"}]}` and answers with JSON.

In [2]:
import json
import time

import httpx

URL = "https://neo4j.het.io/db/data/transaction/commit"
HEADERS = {"Accept": "application/json; charset=UTF-8", "Content-Type": "application/json"}

body = {"statements": [{"statement": "MATCH (n) RETURN count(n) AS nodes"}]}
response = httpx.post(URL, json=body, headers=HEADERS, timeout=60)
time.sleep(0.2)  # be polite: this is a shared public server
payload = response.json()

print("HTTP status   :", response.status_code)
print("top-level keys:", list(payload))
print("payload       :", json.dumps(payload))
print("node count    :", payload["results"][0]["data"][0]["row"][0])

HTTP status   : 200
top-level keys: ['results', 'errors']
payload       : {"results": [{"columns": ["nodes"], "data": [{"row": [47031], "meta": [null]}]}], "errors": []}
node count    : 47031


## 2. The project wrapper: `db.run_cypher`

`run_cypher` makes the same call, retries on network errors, refuses write queries and caps the rows it returns (50 by default). It returns `{"columns", "rows", "total_rows", "truncated"}`, or `{"error"}` when the query is wrong. This function is the only tool the agents get.

In [3]:
import textwrap

from db import run_cypher

labels = run_cypher("CALL db.labels()")
rel_types = run_cypher("CALL db.relationshipTypes()")
print("keys returned by run_cypher:", list(labels))

for title, res in [("node labels", labels), ("relationship types", rel_types)]:
    names = sorted(row[res["columns"][0]] for row in res["rows"])
    print(f"\n{len(names)} {title}:")
    print(textwrap.fill(", ".join(names), 100))

# Write queries are refused locally, before anything is sent to the server.
print("\nwrite attempt ->", run_cypher("CREATE (n:Test) RETURN n"))

keys returned by run_cypher: ['columns', 'rows', 'total_rows', 'truncated']

11 node labels:
Anatomy, BiologicalProcess, CellularComponent, Compound, Disease, Gene, MolecularFunction, Pathway,
PharmacologicClass, SideEffect, Symptom

24 relationship types:
ASSOCIATES_DaG, BINDS_CbG, CAUSES_CcSE, COVARIES_GcG, DOWNREGULATES_AdG, DOWNREGULATES_CdG,
DOWNREGULATES_DdG, EXPRESSES_AeG, INCLUDES_PCiC, INTERACTS_GiG, LOCALIZES_DlA, PALLIATES_CpD,
PARTICIPATES_GpBP, PARTICIPATES_GpCC, PARTICIPATES_GpMF, PARTICIPATES_GpPW, PRESENTS_DpS,
REGULATES_GrG, RESEMBLES_CrC, RESEMBLES_DrD, TREATS_CtD, UPREGULATES_AuG, UPREGULATES_CuG,
UPREGULATES_DuG

write attempt -> {'error': 'Blocked: only read-only Cypher (MATCH/RETURN) is allowed.'}


In [4]:
import pandas as pd

# One hop: which compounds treat multiple sclerosis?
one_hop = """
MATCH (c:Compound)-[:TREATS_CtD]->(d:Disease {name: 'multiple sclerosis'})
RETURN c.name AS compound, c.identifier AS identifier
ORDER BY compound
LIMIT 25
"""
res = run_cypher(one_hop)
print("total_rows:", res["total_rows"], "| truncated:", res["truncated"])
pd.DataFrame(res["rows"]).head(5)

total_rows: 11 | truncated: False


,compound,identifier
0,Azathioprine,DB00993
1,Betamethasone,DB00443
2,Cladribine,DB00242
3,Dexamethasone,DB01234
4,Fingolimod,DB08868


In [5]:
# Two hops with aggregation: the gold query of benchmark question q03.
lines = Path("data/questions.jsonl").read_text(encoding="utf-8-sig").splitlines()
questions = [json.loads(line) for line in lines if line.strip()]
q03 = next(q for q in questions if q["id"] == "q03")
print("question:", textwrap.fill(q03["question"], 100, subsequent_indent=" " * 10))
print("cypher  :", textwrap.fill(q03["gold_cypher"], 100, subsequent_indent=" " * 10))

res = run_cypher(q03["gold_cypher"])
print("same rows as the stored gold_result:", res["rows"] == q03["gold_result"])
pd.DataFrame(res["rows"])

question: Which 5 compounds bind the most genes associated with multiple sclerosis? Return name and number of
          shared genes; break ties alphabetically by name.
cypher  : MATCH (c:Compound)-[:BINDS_CbG]->(g:Gene)<-[:ASSOCIATES_DaG]-(:Disease {name:'multiple sclerosis'})
          RETURN c.name AS name, count(DISTINCT g) AS shared_genes ORDER BY shared_genes DESC, name
          ASC LIMIT 5


same rows as the stored gold_result: True


,name,shared_genes
0,Calcidiol,3
1,Ergocalciferol,3
2,Glucosamine,3
3,Alfacalcidol,2
4,Ampicillin,2


## 3. `EXPLAIN`: check a query without running it

Prefixing a query with `EXPLAIN` asks the server for the execution plan only. `run_cypher` does not pass the plan or the warnings on, so this uses the raw endpoint. The first query is the one-hop pattern from above, which returned 11 rows when run. The second has a label that does not exist in Hetionet (`:Drug` instead of `:Compound`); it is also run once without `EXPLAIN` for comparison.

In [6]:
def post_cypher(statement):
    """Raw POST of one Cypher statement; returns the httpx response."""
    r = httpx.post(URL, json={"statements": [{"statement": statement}]}, headers=HEADERS, timeout=60)
    time.sleep(0.2)
    return r


def plan_operators(node):
    """Flatten the plan tree into a list of operator names, root first."""
    return [node["operatorType"]] + [op for child in node["children"] for op in plan_operators(child)]


valid = ("MATCH (c:Compound)-[:TREATS_CtD]->(d:Disease {name: 'multiple sclerosis'}) "
         "RETURN c.name AS name ORDER BY name")
wrong_label = valid.replace(":Compound", ":Drug")

for title, query in [("valid query", valid), ("unknown label :Drug", wrong_label)]:
    payload = post_cypher("EXPLAIN " + query).json()
    result = payload["results"][0]
    print(f"--- EXPLAIN, {title}")
    print("top-level keys:", list(payload))
    print("result keys   :", list(result))
    print("columns       :", result["columns"], "| rows returned:", len(result["data"]))
    if "plan" in result:
        root = result["plan"]["root"]
        print("plan operators:", " <- ".join(plan_operators(root)))
        print("estimated rows:", round(root["EstimatedRows"], 2))
    print("errors        :", payload["errors"])
    if "notifications" not in payload:
        print("notifications : none (key absent)")
    for note in payload.get("notifications", []):
        print("notification  :", note["code"], f"[{note['severity']}]")
        print("  title       :", note["title"])
        print("  description :", textwrap.fill(note["description"], 100, subsequent_indent=" " * 16))
        print("  position    :", note["position"])
    print()

# The same wrong query run normally, without EXPLAIN.
payload = post_cypher(wrong_label).json()
print("--- run without EXPLAIN, unknown label :Drug")
print("top-level keys:", list(payload))
print("rows returned :", sum(len(res["data"]) for res in payload["results"]), "| errors:", payload["errors"])

--- EXPLAIN, valid query
top-level keys: ['results', 'errors']
result keys   : ['columns', 'data', 'plan']
columns       : ['name'] | rows returned: 0
plan operators: ProduceResults <- Sort <- Projection <- Filter <- Expand(All) <- NodeIndexSeek
estimated rows: 5.51
errors        : []
notifications : none (key absent)



--- EXPLAIN, unknown label :Drug
top-level keys: ['results', 'notifications', 'errors']
result keys   : ['columns', 'data', 'plan']
columns       : ['name'] | rows returned: 0
plan operators: ProduceResults <- Sort <- Projection <- Filter <- Expand(All) <- NodeIndexSeek
estimated rows: 0.0
errors        : []
notification  : Neo.ClientNotification.Statement.UnknownLabelWarning [WARNING]
  title       : The provided label is not in the database.
  description : One of the labels in your query is not available in the database, make sure you didn't misspell it
                or that the label is available when you run this statement in your application (the
                missing label name is: Drug)
  position    : {'offset': 17, 'line': 1, 'column': 18}



--- run without EXPLAIN, unknown label :Drug
top-level keys: ['results', 'errors']
rows returned : 0 | errors: []


## 4. The server speaks Cypher 3.5

Subquery syntax added in Neo4j 4.x is rejected. The server reports this in the `errors` list of a normal HTTP 200 reply, so no exception is raised and the message can be read as text.

In [7]:
newer_syntax = {
    "EXISTS { ... } subquery": (
        "MATCH (c:Compound) WHERE EXISTS { MATCH (c)-[:TREATS_CtD]->(:Disease {name: 'multiple sclerosis'}) } "
        "RETURN c.name AS name LIMIT 5"),
    "CALL { ... } subquery": (
        "MATCH (d:Disease {name: 'multiple sclerosis'}) "
        "CALL { WITH d MATCH (c:Compound)-[:TREATS_CtD]->(d) RETURN count(c) AS n } "
        "RETURN d.name AS disease, n"),
}

for title, query in newer_syntax.items():
    try:
        r = post_cypher(query)
        payload = r.json()
    except (httpx.HTTPError, ValueError) as exc:
        print(f"--- {title}: request failed: {type(exc).__name__}: {exc}\n")
        continue
    print(f"--- {title} | HTTP {r.status_code} | results: {len(payload['results'])} | errors: {len(payload['errors'])}")
    for err in payload["errors"]:
        print("code   :", err["code"])
        print("message:", err["message"])
    print()

--- EXISTS { ... } subquery | HTTP 200 | results: 0 | errors: 1
code   : Neo.ClientError.Statement.SyntaxError
message: Invalid input '(': expected whitespace, comment, ':', ',' or '}' (line 1, column 41 (offset: 40))
"MATCH (c:Compound) WHERE EXISTS { MATCH (c)-[:TREATS_CtD]->(:Disease {name: 'multiple sclerosis'}) } RETURN c.name AS name LIMIT 5"
                                         ^



--- CALL { ... } subquery | HTTP 200 | results: 0 | errors: 1
code   : Neo.ClientError.Statement.SyntaxError
message: Invalid input '{': expected whitespace, comment, namespace of a procedure or a procedure name (line 1, column 53 (offset: 52))
"MATCH (d:Disease {name: 'multiple sclerosis'}) CALL { WITH d MATCH (c:Compound)-[:TREATS_CtD]->(d) RETURN count(c) AS n } RETURN d.name AS disease, n"
                                                     ^



## Takeaways

- One POST per query. The reply has `results` (columns and rows) and `errors`; a Cypher error still arrives with HTTP 200, so `errors` must always be checked.
- `EXPLAIN` returned the plan and no rows. For the wrong label it also returned an `UnknownLabelWarning` notification naming the label; run normally, the same query returned 0 rows with no error and no warning.
- The server accepts Cypher 3.5 only: `EXISTS { }` and `CALL { }` subqueries are syntax errors. Agents that write queries for it must be told the dialect.